# Transfer learning study — feature extraction vs fine-tuning

## What this measures

Three ways to build the same ResNet18 province classifier, on the same 2,515
training crops, for the same 40 epochs, scored on the same 567 held-out crops.
Only the transfer-learning strategy changes.

| run | initial weights | what trains | name |
|---|---|---|---|
| **A** | random | everything | training from scratch (no transfer) |
| **B** | ImageNet | **only the final layer** | **feature extraction** |
| **C** | ImageNet | everything, low LR | **fine-tuning** |

In run B the ImageNet convolutions are frozen and act as a fixed feature
extractor — 11,176,512 parameters locked, 13,338 trainable (0.12% of the model).
In run C every parameter keeps learning.

## What to expect

The usual textbook result is that **feature extraction** wins when the dataset is
small or looks like ImageNet, and **fine-tuning** wins when there is enough data
to adapt the features to a new domain.

Khmer script on license plates looks nothing like ImageNet photos, and 2,515
images is a reasonable amount, so fine-tuning is the likely winner here — but
that is a prediction, not a result. The measure cells decide.

Run time: about 30 minutes total on a T4.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ALPR/alpr_colab_bundle.zip'

import os, zipfile, shutil
shutil.rmtree('/content/alpr', ignore_errors=True)
os.makedirs('/content/alpr', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/alpr')
%cd /content/alpr

In [ ]:
# ---- BUNDLE FRESHNESS CHECK -- do not skip -------------------------------
import glob

tr = open('scripts/recognition/train_province_classifier.py', encoding='utf-8').read()
ok_freeze = '--freeze' in tr and 'FEATURE EXTRACTION' in tr
n_train = len(glob.glob('data/province_crops/train/*/*.jpg'))
n_test  = len(glob.glob('data/province_crops/test/*/*.jpg'))

print('--freeze flag   :', 'PRESENT' if ok_freeze else '*** MISSING ***')
print('province crops  : train', n_train, '| test', n_test)
assert ok_freeze, 'STALE BUNDLE -- rebuild with --crnn-only and re-upload.'
assert n_train >= 2500 and n_test >= 560, 'province crops missing'
print()
print('Bundle is current. Safe to continue.')

In [ ]:
!pip -q install torch torchvision pyyaml tqdm pillow opencv-python-headless

In [ ]:
# ---- RUN A: from scratch (no transfer learning) --------------------------
!python scripts/recognition/train_province_classifier.py --epochs 40 \
    --out models/recognition/prov_A_scratch.pth

In [ ]:
# ---- RUN B: FEATURE EXTRACTION (ImageNet frozen, head only) --------------
# Watch the [freeze] line: it should report ~11.2M frozen, ~13k trainable.
!python scripts/recognition/train_province_classifier.py --pretrained --freeze --epochs 40 \
    --out models/recognition/prov_B_featext.pth

In [ ]:
# ---- RUN C: FINE-TUNING (ImageNet, all layers, low LR) -------------------
!python scripts/recognition/train_province_classifier.py --pretrained --epochs 40 --lr 1e-4 \
    --out models/recognition/prov_C_finetune.pth

In [ ]:
# ---- SCORE ALL THREE on the same held-out crops --------------------------
import subprocess, re

runs = [('A  from scratch      ', 'prov_A_scratch'),
        ('B  feature extraction', 'prov_B_featext'),
        ('C  fine-tuning       ', 'prov_C_finetune')]

print('%-24s %10s %14s' % ('run', 'upright', 'upside-down'))
print('-' * 52)
for label, stem in runs:
    out = subprocess.run(
        ['python', 'scripts/tools/test_province_rotation.py',
         '--weights', f'models/recognition/{stem}.pth',
         '--config',  f'models/recognition/{stem}_config.json'],
        capture_output=True, text=True).stdout
    up = re.search(r'upright\s*:\s*([\d.]+)%', out)
    dn = re.search(r'upside-down\s*:\s*([\d.]+)%', out)
    print('%-24s %9s%% %13s%%' % (label,
                                  up.group(1) if up else '??',
                                  dn.group(1) if dn else '??'))
print()
print('For reference, the DEPLOYED model scores 96.1% upright / 19.4% upside-down.')

### How to read it

Compare the **upright** column — that is the transfer-learning question. (The
upside-down column will be low for all three; none of these runs used
`--rotate180`. That is a separate experiment.)

| what you see | what it means for your report |
|---|---|
| C > B > A | textbook: transfer helps, and adapting the features helps more |
| B > C | the frozen ImageNet features already suit the task; fine-tuning overfit 2,515 images |
| A ≈ C | ImageNet gave nothing — plausible, since Khmer script is far from ImageNet photos |
| A > both | pretrained features actively hurt — unusual, worth reporting |

Any of these is a valid result. The point is that you **measured** the comparison
on your own data instead of asserting the textbook answer.

Paste the whole scoring table back to Claude.

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/ALPR/trained', exist_ok=True)
for stem in ('prov_A_scratch', 'prov_B_featext', 'prov_C_finetune'):
    for ext in ('.pth', '_config.json'):
        p = f'models/recognition/{stem}{ext}'
        if os.path.exists(p):
            shutil.copy(p, '/content/drive/MyDrive/ALPR/trained/')
print('saved to Drive/ALPR/trained')
print('You only need these if you want to deploy one; the numbers are the deliverable.')